In [1]:
from datasets import Dataset
from transformers import BartTokenizer, BartForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback
import torch
import json


c:\All\Maxym\Projects\QA-SLM\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
NVIDIA GeForce RTX 4060 Laptop GPU


In [4]:
rows = []
with open("rag_dataset.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        rows.append({
            "input": item["instruction"], # Тут вже є склеєний Context + Question
            "output": item["response"]
        })

In [5]:
# 2. Створення Dataset
dataset = Dataset.from_list(rows)

# Перевірка
print(dataset[0])

{'input': 'Context: Simple user manual\n•\tFor detailed instructions on installation and cleaning of the appliances, visit the Samsung website \n(http://www.samsung.com), go to Support > Support home, and then enter the model name.\n•\tTo check the product’s model name, see the label enclosed with the product or attached to the \nproduct.\n•\tOpen a QR code scanner app and scan the QR code image attached to the product. You can access \n‘Product registration’, ‘Manual’, and ‘Customer support’.\n•\tFigures and illustrations are provided for reference only and may differ from the actual product \nappearance. Product design and specifications may change without notice.\n\nQuestion: Can I find installation instructions on the Samsung website?', 'output': 'Yes, because the manual says to visit the Samsung website for detailed instructions.'}


In [6]:
# 3. Тренувальний/валідаційний спліт
split_dataset = dataset.train_test_split(test_size=0.15, seed=42)

In [7]:
# 4. Токенізатор і модель
tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")

Loading weights: 100%|██████████| 259/259 [00:00<00:00, 6571.92it/s]


In [8]:
def preprocess(example):
    model_inputs = tokenizer(example["input"], max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(example["output"], max_length=80, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [9]:
tokenized_train = split_dataset["train"].map(preprocess, batched=True)
tokenized_val = split_dataset["test"].map(preprocess, batched=True)

Map: 100%|██████████| 6223/6223 [00:01<00:00, 5364.69 examples/s]


In [2]:
folder_name = "./bart_rag_FT"

In [11]:
# 5. Аргументи навчання
training_args = Seq2SeqTrainingArguments(
    output_dir=folder_name,
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    bf16=True,
    dataloader_num_workers=0,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=25,
    predict_with_generate=True,
    logging_dir="./logs",
    logging_strategy="epoch",
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [12]:
# 6. Тренер
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # Зупинить, якщо 3 епохи loss не падає
)

In [13]:
# 7. Навчання
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.432041,0.246095
2,0.225506,0.180024
3,0.161013,0.151158
4,0.124967,0.136125
5,0.101746,0.131371
6,0.085105,0.129154
7,0.072424,0.127673
8,0.062119,0.129741
9,0.053800,0.131917
10,0.046952,0.132741


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.08it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=44080, training_loss=0.1365673882126159, metrics={'train_runtime': 9706.8844, 'train_samples_per_second': 90.809, 'train_steps_per_second': 11.353, 'total_flos': 1.074934889054208e+17, 'train_loss': 0.1365673882126159, 'epoch': 10.0})

In [14]:
# --- Крок 7: Збереження моделі ---
trainer.save_model(folder_name)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.15it/s]


In [3]:
from sentence_transformers import SentenceTransformer
from transformers import BartTokenizer, BartForConditionalGeneration
import faiss

In [12]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
index = faiss.read_index("../input/KB/refrigerator_kb.index")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15034.92it/s]


In [6]:
def load_chunks_from_file(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Розбиваємо текст по маркеру
    raw_blocks = content.split("🔹 Блок")[1:]
    
    chunks = []
    for block in raw_blocks:
        parts = block.split("-" * 60)
        if len(parts) >= 3:
            chunks.append(parts[1].strip())
    return chunks

In [11]:
chunks = load_chunks_from_file("../input/Instructions/Instruction_v1.4.txt")

In [8]:
model_path = "./bart_rag_FT" # або твій folder_name
tokenizer = BartTokenizer.from_pretrained(model_path)
model = BartForConditionalGeneration.from_pretrained(model_path)

# Переносимо модель на відеокарту для миттєвої генерації
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

Loading weights: 100%|██████████| 260/260 [00:00<00:00, 3445.59it/s]


BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_n

In [9]:
def generate_rag_answer(user_query):
    # Крок А: Пошук релевантного контексту
    query_vector = embedder.encode([user_query], convert_to_numpy=True)
    distances, indices = index.search(query_vector, k=1)
    retrieved_context = chunks[indices[0][0]]
    
    # Крок Б: Формування промпту в тому ж форматі, що і на навчанні
    prompt = f"Context: {retrieved_context}\n\nQuestion: {user_query}"
    
    # Крок В: Генерація відповіді
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_length=80, 
            num_beams=4, # Beam search для більш логічних речень
            early_stopping=True
        )
        
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer, retrieved_context

In [10]:
generated_answers = []
true_answers = []
# 3. Тестуємо систему
question = "What is the optimal temperature for the freezer?"
answer, context = generate_rag_answer(question)

print(f"Питання: {question}")
print(f"Знайдений контекст: {context[:100]}...") 
print(f"Відповідь моделі: {answer}")

Питання: What is the optimal temperature for the freezer?
Знайдений контекст: Appendix
Temperature Instruction
Recommended Temperature
The optimal temperature setting for food st...
Відповідь моделі: The recommended freezer temperature is -19 °C.
